# Chapter 16: Complex Numbers: A Primer

**Source orientation.** Sections 16.1-16.5; printed pages 297-310; PDF pages 319-332. The source span was read for structure, terminology, and concept coverage only. This notebook uses original prose, code, diagrams, and checks.

**Chapter goal.** Build a geometric working model of complex numbers that explains why algebraic closure, roots, rotations, conjugation, and polynomial solving matter for later projective measurement.

**Route.** The chapter starts from the historical pressure created by cubic and quartic equations, then turns complex numbers into points, vectors, rotations, scalings, and reflections. The last sections connect that arithmetic to projective geometry: over `C`, hidden intersections become algebraic data, and field automorphisms such as conjugation become transformations that later chapters must account for.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives-on-Projective-Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-16-complex-numbers-a-primer"
for child in ["figures", "html", "tables", "checks"]:
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

ARTIFACT_ROOT.relative_to(BOOK_ROOT).as_posix()


## Computational Translation Guide

| Source idea | Computational object | What the geometry should protect |
| --- | --- | --- |
| `a + i b` | point `(a, b)` or vector from the origin | real part, imaginary part, distance from origin, and direction |
| length and angle | `abs(z)` and `arg(z)` | multiplication multiplies lengths and adds angles modulo `2*pi` |
| roots | solutions of `w**n = z` | equal angular spacing on a circle, hence a regular polygon |
| Euler formula | sampled path `exp(i*t)` and power series checks | unit length and counterclockwise rotation by parameter `t` |
| conjugation | map `z -> z.conjugate()` | reflection in the real axis and a field automorphism of `C` |
| fundamental theorem of algebra | factorization and root residuals | polynomial roots exist over `C`, even when no real point is visible |
| projective measurement context | homogeneous coordinates over `C` plus circular points later | complex closure and conjugation change what counts as a valid projective map |

The examples below use `matplotlib` for durable 2D geometric diagrams, `plotly` for a parameter lab where angle variation is the point, `numpy` for sampled complex arithmetic, and `sympy` for exact polynomial and intersection checks. Those choices are narrower than a general projective or 3D stack because the chapter is primarily planar complex geometry.


In [ ]:
import cmath
import csv
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import sympy as sp
from matplotlib.patches import Circle, Polygon

from utils.artifacts import assert_artifacts, display_artifact, image_stats, save_json
from utils.projective import cross_ratio

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 170,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

artifact_paths = {}
checks = {}


def rel(path):
    return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()


def fig_path(filename):
    return ARTIFACT_ROOT / "figures" / filename


def html_path(filename):
    return ARTIFACT_ROOT / "html" / filename


def table_path(filename):
    return ARTIFACT_ROOT / "tables" / filename


def save_fig(fig, filename):
    path = fig_path(filename)
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


def xy(z):
    return (float(np.real(z)), float(np.imag(z)))


def draw_complex_arrow(ax, z, color, label=None, origin=0+0j, lw=2.0, alpha=1.0):
    ox, oy = xy(origin)
    dx, dy = xy(z - origin)
    ax.arrow(ox, oy, dx, dy, length_includes_head=True, head_width=0.055, head_length=0.09,
             linewidth=lw, color=color, alpha=alpha)
    if label:
        tx, ty = xy(z)
        ax.text(tx + 0.06, ty + 0.06, label, color=color, weight="bold")


def setup_complex_axes(ax, lim=2.2, title=None):
    ax.axhline(0, color="#444444", linewidth=0.8)
    ax.axvline(0, color="#444444", linewidth=0.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.grid(True, color="#e5e7eb", linewidth=0.7)
    ax.set_xlabel("real")
    ax.set_ylabel("imaginary")
    if title:
        ax.set_title(title)


def principal_angle(theta):
    return float((theta + math.pi) % (2 * math.pi) - math.pi)


## Refreshed Visual Storyboard

The earlier chapter scaffold treated this primer as if it were already about Mobius motion on the complex projective line. That belongs more naturally to the next chapter. This refreshed storyboard keeps Chapter 16 at the primer level while still pointing toward projective measurement.


In [ ]:
storyboard = {
    "chapter goal": "Build a geometric working model of complex numbers for projective measurement chapters.",
    "source span read": "sections 16.1-16.5; printed pages 297-310; PDF pages 319-332",
    "concept inventory": [
        "cubic equations historically forced imaginary intermediate quantities",
        "quartic formulas depend on the same polynomial-solving culture and on cubic resolvents",
        "the fundamental theorem of algebra lets geometric intersections be counted over C",
        "a complex number is a point with length and angle",
        "multiplication is a rotation combined with a stretch",
        "nth roots form a regular polygon in the complex plane",
        "Euler's formula parameterizes the unit circle by exp(i*t)",
        "conjugation is reflection and a nontrivial field automorphism",
        "complex closure and conjugation are projective-geometry issues, not only algebraic syntax",
    ],
    "library routing table": [
        {"concept": "Argand arithmetic, roots, conjugation", "representation": "labeled 2D complex-plane diagrams", "library": "matplotlib", "why": "durable static geometry with equal aspect and explicit vectors", "fallback": "plain SVG"},
        {"concept": "angle variation under multiplication", "representation": "slider over multiplier angle", "library": "plotly", "why": "standalone HTML keeps the parameter sweep inspectable outside a live kernel", "fallback": "sampled Matplotlib panels"},
        {"concept": "polynomial identities and hidden intersections", "representation": "exact residual checks", "library": "sympy", "why": "small exact identities expose algebraic closure without numerical guesswork", "fallback": "numpy residuals with tolerance"},
        {"concept": "root and angle residuals", "representation": "sampled numeric diagnostics", "library": "numpy", "why": "complex arithmetic maps directly to the plane", "fallback": "Python complex numbers"},
    ],
    "visual sequence": [
        {"concept": "cubic and quartic root ledger", "artifact": "figures/cubic-quartic-root-ledger.png", "inspection target": "imaginary Cardano pieces cancel to a real cubic root; quartic roots occur as conjugate pairs", "validation": "Cardano cube and polynomial residuals"},
        {"concept": "complex closure of conic-line intersections", "artifact": "figures/projective-complex-intersections.png", "inspection target": "a line missing a real circle still has two complex intersection roots", "validation": "exact conic-line residuals"},
        {"concept": "length and angle multiplication", "artifact": "figures/angle-length-multiplication.png", "inspection target": "product vector is the old vector rotated and stretched", "validation": "modulus product and angle sum residuals"},
        {"concept": "roots as a regular polygon", "artifact": "figures/roots-regular-polygon.png", "inspection target": "all roots lie on one circle with equal angle spacing", "validation": "root power residual and edge-length spread"},
        {"concept": "Euler rotation sweep", "artifact": "figures/euler-rotation-sweep.png", "inspection target": "exp(i*t) runs around the unit circle and Taylor approximations converge", "validation": "Euler identity and series residuals"},
        {"concept": "conjugation as reflection and automorphism", "artifact": "figures/conjugation-reflection-automorphism.png", "inspection target": "conjugation flips orientation but preserves addition and multiplication laws", "validation": "automorphism and real-polynomial conjugate-root residuals"},
        {"concept": "angle-length parameter lab", "artifact": "html/angle-length-root-lab.html", "inspection target": "slider shows angle addition while length scale stays visible", "validation": "HTML artifact exists and records sampled frames"},
        {"concept": "root diagnostics table", "artifact": "tables/root-angle-length-diagnostics.csv", "inspection target": "numeric tolerances for several root polygons", "validation": "all residual columns stay below tolerance"},
    ],
    "computational checks": [
        "artifact existence and nonzero size",
        "nonblank raster statistics for every generated PNG",
        "Cardano cubic residual and quartic root residuals",
        "line-conic complex intersection residuals",
        "multiplication length and angle identities",
        "root polygon equal-angle and equal-edge diagnostics",
        "Euler formula and Taylor-series residuals",
        "conjugation field-automorphism identities",
        "cross-ratio invariance over complex samples as a projective bridge",
    ],
    "proof-visualization strategy": "Use invariant ledgers: every visual has a nearby residual or exact identity rather than a copied proof.",
    "implementation notes": "All artifacts are written under artifacts/chapter-16-complex-numbers-a-primer with relative paths recorded in JSON.",
    "risks": ["The chapter is a primer, so projective measurement is previewed rather than fully developed.", "Plotly HTML interactivity is saved as a standalone artifact for static notebook exports."],
    "acceptance criteria": ["Chapter 16 executes with nbclient", "all named artifacts exist and are nonzero", "final-sanity.json records numeric invariants below tolerance"],
}
artifact_paths["storyboard"] = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
storyboard["visual sequence"]


## Historical Pressure: Cubics, Quartics, and Invisible Detours

Complex numbers did not enter mathematics because someone wanted a new coordinate plane. They became unavoidable when algebraic formulas for polynomial equations led through quantities that were not real, even when the final answer was real. A standard cubic example is `x**3 - 15*x - 4 = 0`: one real root is produced by adding two conjugate complex cube roots.

The quartic story belongs to the same pipeline. Once cubics had formulas, quartic equations could be reduced to a cubic auxiliary problem. The lesson for geometry is not the formula itself; it is that polynomial solving wants a field where intermediate and final roots can be counted without changing rules midstream.


In [ ]:
x = sp.symbols("x")
cardano_poly = x**3 - 15*x - 4
u = 2 + 1j
v = 2 - 1j
cardano_root = u + v
cardano_roots = sorted([complex(r) for r in np.roots([1, 0, -15, -4])], key=lambda z: z.real)

quartic_roots_exact = [2 + 1j, 2 - 1j, -1 + 2j, -1 - 2j]
quartic_coeffs = np.poly(quartic_roots_exact).real
quartic_poly = sum(sp.Integer(round(c)) * x**(4 - i) for i, c in enumerate(quartic_coeffs))

fig, axes = plt.subplots(1, 3, figsize=(13.4, 4.0))
xs = np.linspace(-4.4, 4.5, 500)
ys = xs**3 - 15 * xs - 4
axes[0].plot(xs, ys, color="#1f77b4", linewidth=2)
axes[0].axhline(0, color="#333333", linewidth=0.8)
for r in cardano_roots:
    axes[0].scatter([r.real], [0], s=38, color="#c2410c", zorder=3)
    axes[0].text(r.real, 5.5, f"{r.real:.3g}", ha="center", color="#7c2d12")
axes[0].set_title("Cubic with three real roots")
axes[0].set_xlabel("x")
axes[0].set_ylabel("x^3 - 15x - 4")
axes[0].set_ylim(-30, 30)
axes[0].grid(True, color="#e5e7eb")

setup_complex_axes(axes[1], lim=4.4, title="Cardano pieces in the complex plane")
draw_complex_arrow(axes[1], u, "#2563eb", "u = 2+i")
draw_complex_arrow(axes[1], v, "#0f766e", "v = 2-i")
draw_complex_arrow(axes[1], cardano_root, "#b45309", "u+v = 4", lw=2.4)
axes[1].plot([u.real, v.real], [u.imag, v.imag], linestyle="--", color="#94a3b8")
axes[1].text(-3.95, -3.65, "complex detour, real sum", color="#475569")

setup_complex_axes(axes[2], lim=3.6, title="Quartic roots as conjugate pairs")
colors = ["#9333ea", "#9333ea", "#16a34a", "#16a34a"]
for root, color in zip(quartic_roots_exact, colors):
    axes[2].scatter([root.real], [root.imag], s=54, color=color)
    axes[2].text(root.real + 0.08, root.imag + 0.08, f"{root.real:g}{root.imag:+g}i", color=color)
axes[2].plot([2, 2], [-1, 1], color="#c4b5fd", linestyle="--")
axes[2].plot([-1, -1], [-2, 2], color="#bbf7d0", linestyle="--")
axes[2].text(-3.35, -3.05, f"quartic: {sp.expand(quartic_poly)}", fontsize=8, color="#334155")

fig.suptitle("Polynomial solving pushed arithmetic beyond the real line", y=1.02, fontsize=13, weight="bold")
artifact_paths["cubic_quartic"] = save_fig(fig, "cubic-quartic-root-ledger.png")

checks["cardano_u_cube_residual"] = float(abs(u**3 - (2 + 11j)))
checks["cardano_v_cube_residual"] = float(abs(v**3 - (2 - 11j)))
checks["cardano_real_root_residual"] = float(abs(cardano_root**3 - 15 * cardano_root - 4))
checks["quartic_integer_coefficients"] = [int(round(c)) for c in quartic_coeffs]
checks["quartic_root_max_residual"] = float(max(abs(np.polyval(quartic_coeffs, r)) for r in quartic_roots_exact))
rel(artifact_paths["cubic_quartic"]), checks["cardano_real_root_residual"], checks["quartic_integer_coefficients"]


## Algebraic Closure as a Projective Habit

The fundamental theorem of algebra says that a degree `n` polynomial has `n` complex roots when multiplicity is counted. In projective geometry this becomes a practical habit: intersections may be absent from the real drawing but still present in the algebra.

The figure below uses the circle `x**2 + y**2 = 1`. A vertical line with `x = 0.65` visibly meets the circle in two real points. A vertical line with `x = 1.35` has no real intersection, but the equation still has two complex solutions for `y`. This is the same bookkeeping principle later used for conics, circular points, and projective measurement structures.


In [ ]:
y = sp.symbols("y")
x_real = sp.Rational(13, 20)
x_hidden = sp.Rational(27, 20)
real_y_roots = sp.solve(sp.Eq(x_real**2 + y**2, 1), y)
hidden_y_roots = sp.solve(sp.Eq(x_hidden**2 + y**2, 1), y)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.3))
theta = np.linspace(0, 2 * np.pi, 500)
axes[0].plot(np.cos(theta), np.sin(theta), color="#1d4ed8", linewidth=2, label="x^2+y^2=1")
for xval, color, label in [(float(x_real), "#15803d", "two real intersections"), (float(x_hidden), "#dc2626", "two complex intersections")]:
    axes[0].plot([xval, xval], [-1.25, 1.25], color=color, linewidth=2, label=label)
for yr in real_y_roots:
    axes[0].scatter([float(x_real)], [float(yr)], color="#15803d", s=48, zorder=3)
axes[0].text(float(x_hidden) + 0.04, 0.2, "no real point\non the circle", color="#991b1b")
axes[0].set_aspect("equal")
axes[0].set_xlim(-1.35, 1.65)
axes[0].set_ylim(-1.3, 1.3)
axes[0].grid(True, color="#e5e7eb")
axes[0].legend(loc="lower left", fontsize=8)
axes[0].set_title("Real drawing")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")

setup_complex_axes(axes[1], lim=1.55, title="Complex y-values for x = 1.35")
for root in hidden_y_roots:
    z = complex(sp.N(root))
    axes[1].scatter([z.real], [z.imag], s=62, color="#dc2626")
    axes[1].text(z.real + 0.05, z.imag + 0.05, f"y={z.real:.1f}{z.imag:+.3f}i", color="#991b1b")
axes[1].text(-1.42, -1.32, "The line-conic count survives over C.", color="#334155")

artifact_paths["complex_intersections"] = save_fig(fig, "projective-complex-intersections.png")
hidden_residuals = [sp.simplify(x_hidden**2 + root**2 - 1) for root in hidden_y_roots]
real_residuals = [sp.simplify(x_real**2 + root**2 - 1) for root in real_y_roots]
checks["hidden_line_conic_exact_residuals"] = [str(r) for r in hidden_residuals]
checks["real_line_conic_exact_residuals"] = [str(r) for r in real_residuals]
rel(artifact_paths["complex_intersections"]), hidden_y_roots


## Length and Angle: Multiplication as a Similarity

Addition of complex numbers is vector addition. Multiplication is more structured: if `z = r*exp(i*theta)` and `w = s*exp(i*phi)`, then `z*w = r*s*exp(i*(theta+phi))`. The real two-by-two matrix of multiplication by `w` is a rotation by `phi` followed by a stretch by `s`.


In [ ]:
z1 = 1.25 + 0.65j
z2 = -0.35 + 0.95j
w = 0.92 * np.exp(1j * np.deg2rad(55))
product = z1 * w

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.5))
setup_complex_axes(axes[0], lim=2.3, title="Addition is vector translation")
draw_complex_arrow(axes[0], z1, "#2563eb", "z")
draw_complex_arrow(axes[0], z2, "#0f766e", "w")
draw_complex_arrow(axes[0], z1 + z2, "#b45309", "z+w", lw=2.4)
axes[0].plot([z1.real, (z1 + z2).real], [z1.imag, (z1 + z2).imag], color="#0f766e", linestyle="--")
axes[0].plot([z2.real, (z1 + z2).real], [z2.imag, (z1 + z2).imag], color="#2563eb", linestyle="--")

setup_complex_axes(axes[1], lim=2.3, title="Multiplication rotates and stretches")
axes[1].add_patch(Circle((0, 0), 1, fill=False, color="#cbd5e1", linewidth=1.2))
draw_complex_arrow(axes[1], z1, "#2563eb", "z")
draw_complex_arrow(axes[1], w, "#0f766e", "multiplier")
draw_complex_arrow(axes[1], product, "#b45309", "z*multiplier", lw=2.4)
angle_arc = np.linspace(cmath.phase(z1), cmath.phase(product), 80)
axes[1].plot(0.55 * np.cos(angle_arc), 0.55 * np.sin(angle_arc), color="#b45309", linewidth=2)
axes[1].text(-2.08, -1.95, "lengths multiply; angles add", color="#334155")

artifact_paths["angle_length"] = save_fig(fig, "angle-length-multiplication.png")
length_residual = abs(abs(product) - abs(z1) * abs(w))
angle_residual = abs(principal_angle(cmath.phase(product) - cmath.phase(z1) - cmath.phase(w)))
matrix = abs(w) * np.array([[np.cos(cmath.phase(w)), -np.sin(cmath.phase(w))], [np.sin(cmath.phase(w)), np.cos(cmath.phase(w))]])
mat_product = matrix @ np.array([z1.real, z1.imag])
checks["multiplication_length_residual"] = float(length_residual)
checks["multiplication_angle_residual"] = float(angle_residual)
checks["multiplication_matrix_residual"] = float(np.linalg.norm(mat_product - np.array([product.real, product.imag])))
rel(artifact_paths["angle_length"]), checks["multiplication_angle_residual"]


## Roots as Regular Polygons

If `root**n = target`, then the length of each root is the `n`th root of `abs(target)`, and the angles differ by `2*pi/n`. The roots are therefore vertices of a regular polygon centered at the origin. This is the geometric content of polar form: solving a power equation is angle division plus a choice of branch.


In [ ]:
n = 5
target = 1.45 * np.exp(1j * np.deg2rad(70))
radius = abs(target) ** (1 / n)
base_angle = cmath.phase(target) / n
roots = np.array([radius * np.exp(1j * (base_angle + 2 * np.pi * k / n)) for k in range(n)])
root_angles = np.unwrap(np.angle(roots))
angle_spacings = np.diff(np.r_[root_angles, root_angles[0] + 2 * np.pi])
edge_lengths = np.abs(np.roll(roots, -1) - roots)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.5))
setup_complex_axes(axes[0], lim=1.75, title="Five roots of one target")
axes[0].add_patch(Circle((0, 0), radius, fill=False, color="#cbd5e1", linewidth=1.2))
polygon = Polygon(np.column_stack([roots.real, roots.imag]), closed=True, fill=True, alpha=0.15, color="#8b5cf6")
axes[0].add_patch(polygon)
for k, root in enumerate(roots):
    draw_complex_arrow(axes[0], root, "#7c3aed", f"r{k}", lw=1.5)
axes[0].text(-1.62, -1.47, "equal spacing = 2*pi/n", color="#334155")

setup_complex_axes(axes[1], lim=1.8, title="Power map sends each root to target")
axes[1].add_patch(Circle((0, 0), abs(target), fill=False, color="#fde68a", linewidth=1.2))
draw_complex_arrow(axes[1], target, "#b45309", "target")
images = roots**n
for image in images:
    axes[1].scatter([image.real], [image.imag], s=42, color="#b45309", alpha=0.55)
for root in roots:
    axes[1].plot([root.real, target.real], [root.imag, target.imag], color="#d8b4fe", linewidth=0.8, alpha=0.7)
axes[1].text(-1.65, -1.52, "all n powers agree", color="#334155")

artifact_paths["roots_polygon"] = save_fig(fig, "roots-regular-polygon.png")
checks["root_power_max_residual"] = float(np.max(np.abs(images - target)))
checks["root_angle_spacing_spread"] = float(np.max(angle_spacings) - np.min(angle_spacings))
checks["root_edge_length_spread"] = float(np.max(edge_lengths) - np.min(edge_lengths))
rel(artifact_paths["roots_polygon"]), checks["root_power_max_residual"]


## Euler's Formula as a Rotation Sweep

Euler's formula `exp(i*t) = cos(t) + i*sin(t)` is a compact way to say that the exponential of a purely imaginary input moves on the unit circle. The formula also explains why angle is periodic: `t` and `t + 2*pi` name the same point on the circle.

The second panel below is not a proof of the infinite series identity, but it makes the power-series route inspectable: as more terms of the exponential series are included, the sampled approximation converges to the circle parameterization.


In [ ]:
ts = np.linspace(0, 2 * np.pi, 361)
euler_points = np.exp(1j * ts)

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.5))
setup_complex_axes(axes[0], lim=1.25, title="exp(i*t) traces the unit circle")
axes[0].plot(euler_points.real, euler_points.imag, color="#2563eb", linewidth=2)
for frac, label in [(0, "0"), (0.25, "pi/2"), (0.5, "pi"), (0.75, "3pi/2"), (1.0, "2pi")]:
    idx = int(frac * (len(ts) - 1))
    z = euler_points[idx]
    axes[0].scatter([z.real], [z.imag], color="#dc2626", s=38)
    axes[0].text(z.real + 0.04, z.imag + 0.04, label, color="#991b1b")

orders = [3, 5, 9, 17]
for order in orders:
    approx = np.zeros_like(euler_points, dtype=complex)
    for k in range(order):
        approx += (1j * ts) ** k / math.factorial(k)
    err = np.abs(approx - euler_points)
    axes[1].plot(ts, err, label=f"{order} terms")
axes[1].set_yscale("log")
axes[1].set_xlabel("t")
axes[1].set_ylabel("absolute error")
axes[1].set_title("Truncated exp series error")
axes[1].grid(True, color="#e5e7eb")
axes[1].legend(fontsize=8)

artifact_paths["euler_sweep"] = save_fig(fig, "euler-rotation-sweep.png")
series_order = 11
sx = sp.symbols("x")
series_exp = sp.series(sp.exp(sp.I * sx), sx, 0, series_order).removeO()
series_split = sp.series(sp.cos(sx), sx, 0, series_order).removeO() + sp.I * sp.series(sp.sin(sx), sx, 0, series_order).removeO()
checks["euler_identity_max_residual"] = float(np.max(np.abs(euler_points - (np.cos(ts) + 1j * np.sin(ts)))))
checks["euler_series_symbolic_residual"] = str(sp.expand(series_exp - series_split))
checks["euler_unit_circle_max_residual"] = float(np.max(np.abs(np.abs(euler_points) - 1)))
rel(artifact_paths["euler_sweep"]), checks["euler_identity_max_residual"]


## Conjugation: Reflection and Field Automorphism

Complex conjugation sends `a+i*b` to `a-i*b`. Geometrically it is reflection in the real axis. Algebraically it preserves addition and multiplication, so it is a field automorphism. That combination matters for projective geometry over `C`: some incidence-preserving maps can include conjugation, not only matrix multiplication with complex entries.

For real-coefficient polynomials, conjugation gives a useful root rule: if `z` is a root, then `conj(z)` is also a root.


In [ ]:
z = 1.25 + 0.85j
w2 = -0.55 + 0.75j
poly_coeffs = np.poly([z, z.conjugate(), -0.4]).real
poly_roots = np.roots(poly_coeffs)

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.4))
setup_complex_axes(axes[0], lim=1.8, title="Conjugation is reflection")
draw_complex_arrow(axes[0], z, "#2563eb", "z")
draw_complex_arrow(axes[0], z.conjugate(), "#dc2626", "conj(z)")
axes[0].plot([z.real, z.real], [z.imag, -z.imag], linestyle="--", color="#94a3b8")
axes[0].scatter([z.real], [0], color="#475569", s=32)
axes[0].text(-1.65, -1.52, "real axis is fixed pointwise", color="#334155")

setup_complex_axes(axes[1], lim=1.8, title="Real polynomial roots pair by conjugation")
for root in poly_roots:
    color = "#9333ea" if abs(root.imag) > 1e-8 else "#0f766e"
    axes[1].scatter([root.real], [root.imag], s=58, color=color)
    axes[1].text(root.real + 0.05, root.imag + 0.06, f"{root.real:.2f}{root.imag:+.2f}i", color=color)
axes[1].text(-1.65, -1.52, "real coefficients force mirror roots", color="#334155")

artifact_paths["conjugation"] = save_fig(fig, "conjugation-reflection-automorphism.png")
checks["conjugation_addition_residual"] = float(abs((z + w2).conjugate() - (z.conjugate() + w2.conjugate())))
checks["conjugation_multiplication_residual"] = float(abs((z * w2).conjugate() - (z.conjugate() * w2.conjugate())))
checks["conjugation_norm_residual"] = float(abs(z * z.conjugate() - abs(z) ** 2))
checks["real_polynomial_conjugate_root_residual"] = float(abs(np.polyval(poly_coeffs, z.conjugate())))
rel(artifact_paths["conjugation"]), checks["conjugation_multiplication_residual"]


## Interactive Angle-Length Lab

The static multiplication diagram shows one product. The HTML lab below keeps the first complex number fixed and varies the angle of the multiplier. Move the slider to inspect two facts at once: the product rotates by the added angle, while the product length remains the product of the two input lengths.


In [ ]:
fixed_z = 0.95 + 0.35j
scale = 1.2
angles = np.linspace(0, 2 * np.pi, 37)
circle_x = np.cos(np.linspace(0, 2 * np.pi, 300))
circle_y = np.sin(np.linspace(0, 2 * np.pi, 300))

base_traces = [
    go.Scatter(x=circle_x, y=circle_y, mode="lines", line=dict(color="#cbd5e1"), name="unit circle"),
    go.Scatter(x=[0, fixed_z.real], y=[0, fixed_z.imag], mode="lines+markers+text", text=["", "z"], textposition="top right", line=dict(color="#2563eb", width=4), marker=dict(size=8), name="fixed z"),
]
frames = []
for theta in angles:
    multiplier = scale * np.exp(1j * theta)
    product = fixed_z * multiplier
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[0, multiplier.real], y=[0, multiplier.imag], mode="lines+markers+text", text=["", "m"], textposition="top right", line=dict(color="#0f766e", width=4), marker=dict(size=8), name="multiplier"),
            go.Scatter(x=[0, product.real], y=[0, product.imag], mode="lines+markers+text", text=["", "z*m"], textposition="top right", line=dict(color="#b45309", width=4), marker=dict(size=9), name="product"),
        ],
        name=f"{np.rad2deg(theta):.0f} deg",
    ))

fig = go.Figure(data=base_traces + list(frames[0].data), frames=frames)
fig.update_layout(
    title="Complex multiplication: angle slider",
    xaxis=dict(scaleanchor="y", scaleratio=1, range=[-1.9, 1.9], zeroline=True, gridcolor="#e5e7eb", title="real"),
    yaxis=dict(range=[-1.9, 1.9], zeroline=True, gridcolor="#e5e7eb", title="imaginary"),
    width=780,
    height=620,
    margin=dict(l=40, r=20, t=60, b=50),
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "multiplier angle: "},
        "steps": [{"label": frame.name, "method": "animate", "args": [[frame.name], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}]} for frame in frames],
    }],
    updatemenus=[{"type": "buttons", "showactive": False, "buttons": [{"label": "play", "method": "animate", "args": [None, {"frame": {"duration": 110, "redraw": True}, "fromcurrent": True}]}, {"label": "pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]}]}],
)
path_html = html_path("angle-length-root-lab.html")
fig.write_html(path_html, include_plotlyjs="cdn", full_html=True)
artifact_paths["angle_lab"] = path_html
checks["angle_lab_frame_count"] = len(frames)
checks["angle_lab_scale"] = float(scale)
rel(path_html), path_html.stat().st_size


## Applied Lab: Root Diagnostics Across Several Orders

The lab repeats the root-polygon construction for several values of `n`. The useful question is not whether the picture looks regular; it is which quantities certify regularity: root-power residual, angular spacing spread, and edge-length spread.


In [ ]:
def root_polygon_diagnostics(n, target):
    radius = abs(target) ** (1 / n)
    base = cmath.phase(target) / n
    roots = np.array([radius * np.exp(1j * (base + 2 * np.pi * k / n)) for k in range(n)])
    angles = np.unwrap(np.angle(roots))
    spacings = np.diff(np.r_[angles, angles[0] + 2 * np.pi])
    edges = np.abs(np.roll(roots, -1) - roots)
    return {
        "n": n,
        "target_real": float(target.real),
        "target_imag": float(target.imag),
        "root_power_max_residual": float(np.max(np.abs(roots**n - target))),
        "angle_spacing_spread": float(np.max(spacings) - np.min(spacings)),
        "edge_length_spread": float(np.max(edges) - np.min(edges)),
        "root_radius": float(radius),
    }

lab_rows = [root_polygon_diagnostics(n, 1.1 * np.exp(1j * np.deg2rad(25 + 7 * n))) for n in range(3, 9)]
path_table = table_path("root-angle-length-diagnostics.csv")
with path_table.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(lab_rows[0].keys()))
    writer.writeheader()
    writer.writerows(lab_rows)
artifact_paths["root_diagnostics_table"] = path_table
checks["root_lab_max_power_residual"] = float(max(row["root_power_max_residual"] for row in lab_rows))
checks["root_lab_max_angle_spacing_spread"] = float(max(row["angle_spacing_spread"] for row in lab_rows))
checks["root_lab_max_edge_length_spread"] = float(max(row["edge_length_spread"] for row in lab_rows))
lab_rows


## Projective Measurement Bridge

This chapter is still a primer, but it sets up later measurement chapters in two precise ways.

First, projective constructions over `C` can count intersections uniformly. The real plane may hide a conic intersection, while the complex calculation still records the two roots. Second, conjugation is a field automorphism. The fundamental theorem of projective geometry therefore has an extra branch over `C`: some incidence-preserving maps are semilinear because they combine a matrix with conjugation.

A compact computational bridge is cross-ratio invariance. The cross-ratio works over complex coordinates just as over real coordinates, and Mobius transformations preserve it as long as the four points avoid poles and coincidences.


In [ ]:
complex_points = [0.2 + 0.4j, 1.1 - 0.15j, -0.35 + 0.9j, 0.75 + 1.2j]
a, b, c, d = complex_points
A, B, C, D = 1 + 0.3j, -0.2 + 0.4j, 0.15 - 0.1j, 0.9 - 0.25j
mobius = lambda z0: (A * z0 + B) / (C * z0 + D)
image_points = [mobius(z0) for z0 in complex_points]
cr0 = cross_ratio(a, b, c, d)
cr1 = cross_ratio(*image_points)
checks["complex_cross_ratio_residual"] = float(abs(cr0 - cr1))
checks["complex_mobius_determinant_abs"] = float(abs(A * D - B * C))
{"cross_ratio_before": cr0, "cross_ratio_after": cr1, "residual": checks["complex_cross_ratio_residual"]}


## Artifact Gallery

The notebook displays generated artifacts inline when executed. The direct links below keep the lesson readable in static form.

![Cubic and quartic root ledger](../../artifacts/chapter-16-complex-numbers-a-primer/figures/cubic-quartic-root-ledger.png)

![Complex conic-line intersections](../../artifacts/chapter-16-complex-numbers-a-primer/figures/projective-complex-intersections.png)

![Angle and length multiplication](../../artifacts/chapter-16-complex-numbers-a-primer/figures/angle-length-multiplication.png)

![Roots as a regular polygon](../../artifacts/chapter-16-complex-numbers-a-primer/figures/roots-regular-polygon.png)

![Euler rotation sweep](../../artifacts/chapter-16-complex-numbers-a-primer/figures/euler-rotation-sweep.png)

![Conjugation as reflection](../../artifacts/chapter-16-complex-numbers-a-primer/figures/conjugation-reflection-automorphism.png)

Open the [interactive angle-length lab](../../artifacts/chapter-16-complex-numbers-a-primer/html/angle-length-root-lab.html) and the [root diagnostics table](../../artifacts/chapter-16-complex-numbers-a-primer/tables/root-angle-length-diagnostics.csv) when reading outside an executed kernel.


In [ ]:
for key in ["cubic_quartic", "complex_intersections", "angle_length", "roots_polygon", "euler_sweep", "conjugation", "angle_lab", "root_diagnostics_table"]:
    display_artifact(artifact_paths[key], width=760, height=460)


## Takeaways

- Complex numbers can be read geometrically as points with length and angle.
- Multiplication is a similarity: lengths multiply and angles add.
- The roots of `w**n = z` form a regular polygon because polar angles split into equal branches.
- Euler's formula turns angle into exponential notation and explains the periodicity of rotation.
- Conjugation is both reflection and a field automorphism, so complex projective geometry has semilinear behavior that real projective geometry does not.
- Algebraic closure is a measurement tool: it lets projective geometry count intersections and roots even when the real picture hides them.


In [ ]:
png_keys = ["cubic_quartic", "complex_intersections", "angle_length", "roots_polygon", "euler_sweep", "conjugation"]
html_keys = ["angle_lab"]
table_keys = ["root_diagnostics_table"]
check_keys = ["storyboard"]
all_artifacts = [artifact_paths[k] for k in png_keys + html_keys + table_keys + check_keys]
assert_artifacts(all_artifacts)

raster_stats = [image_stats(artifact_paths[k]) for k in png_keys]
for stat in raster_stats:
    assert stat["width"] >= 200 and stat["height"] >= 150
    assert stat["pixel_std"] > 1.0

assert checks["cardano_u_cube_residual"] < 1e-12
assert checks["cardano_v_cube_residual"] < 1e-12
assert checks["cardano_real_root_residual"] < 1e-12
assert checks["quartic_root_max_residual"] < 1e-10
assert checks["hidden_line_conic_exact_residuals"] == ["0", "0"]
assert checks["real_line_conic_exact_residuals"] == ["0", "0"]
assert checks["multiplication_length_residual"] < 1e-12
assert checks["multiplication_angle_residual"] < 1e-12
assert checks["multiplication_matrix_residual"] < 1e-12
assert checks["root_power_max_residual"] < 1e-12
assert checks["root_angle_spacing_spread"] < 1e-12
assert checks["root_edge_length_spread"] < 1e-12
assert checks["euler_identity_max_residual"] < 1e-12
assert checks["euler_series_symbolic_residual"] == "0"
assert checks["euler_unit_circle_max_residual"] < 1e-12
assert checks["conjugation_addition_residual"] < 1e-12
assert checks["conjugation_multiplication_residual"] < 1e-12
assert checks["conjugation_norm_residual"] < 1e-12
assert checks["real_polynomial_conjugate_root_residual"] < 1e-10
assert checks["root_lab_max_power_residual"] < 1e-10
assert checks["root_lab_max_angle_spacing_spread"] < 1e-12
assert checks["root_lab_max_edge_length_spread"] < 1e-12
assert checks["complex_cross_ratio_residual"] < 1e-12
assert checks["angle_lab_frame_count"] >= 24

visual_checks = {
    "chapter": 16,
    "all_files_exist": all(path.exists() and path.stat().st_size > 256 for path in all_artifacts),
    "raster_artifacts": [{**stat, "path": rel(stat["path"])} for stat in raster_stats],
    "html_artifact": rel(artifact_paths["angle_lab"]),
    "table_artifact": rel(artifact_paths["root_diagnostics_table"]),
    "visual_count": len(png_keys) + len(html_keys) + len(table_keys),
    "numeric_checks": checks,
}
artifact_paths["visual_checks"] = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final = {
    "chapter": 16,
    "title": "Complex Numbers: A Primer",
    "source_span": "sections 16.1-16.5; printed pages 297-310; PDF pages 319-332",
    "artifacts": [rel(path) for path in all_artifacts] + [rel(artifact_paths["visual_checks"])],
    "checks": checks,
    "raster_artifact_count": len(raster_stats),
    "notebook_executed": True,
}
artifact_paths["final_sanity"] = save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
assert_artifacts([artifact_paths["visual_checks"], artifact_paths["final_sanity"]])
final
